In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. LOAD DATA
# ============================================================

INPUT_PATH = "outputs_new/parent/dedupe_clusters_long_full.parquet"

OUT_DIR = Path("outputs_new/server_dedupe_impact")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INPUT_PATH)

# ============================================================
# 2. BASIC CLEANING
# ============================================================

required_cols = [
    "record_id",
    "server_name",
    "parent_server_name_selected",
    "records_hierarchy",
    "is_parent_selected",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["server_name"] = df["server_name"].fillna("Unknown").astype(str)
df["parent_server_name_selected"] = (
    df["parent_server_name_selected"].fillna("Unknown").astype(str)
)
df["records_hierarchy"] = df["records_hierarchy"].fillna("Unknown").astype(str)

# Normalize boolean parent column if needed
df["is_parent_selected"] = df["is_parent_selected"].astype(bool)

# ============================================================
# 3. CLASSIFY RECORD BEHAVIOR
# ============================================================

df["dedupe_behavior"] = np.select(
    [
        (df["server_name"] == df["parent_server_name_selected"])
        & (df["is_parent_selected"] == True),

        (df["server_name"] == df["parent_server_name_selected"])
        & (df["is_parent_selected"] == False),

        (df["server_name"] != df["parent_server_name_selected"]),
    ],
    [
        "stable_parent",
        "internal_duplicate_or_version",
        "cross_server_reassigned",
    ],
    default="unclassified",
)

# ============================================================
# 4. BEFORE / AFTER COUNTS
# ============================================================

before_counts = (
    df.groupby("server_name")
    .size()
    .rename("records_before_deduplication")
)

after_counts = (
    df[df["is_parent_selected"] == True]
    .groupby("parent_server_name_selected")
    .size()
    .rename("parent_records_after_deduplication")
)

stable_parent_counts = (
    df[df["dedupe_behavior"] == "stable_parent"]
    .groupby("server_name")
    .size()
    .rename("stable_parent_records")
)

internal_duplicate_counts = (
    df[df["dedupe_behavior"] == "internal_duplicate_or_version"]
    .groupby("server_name")
    .size()
    .rename("internal_duplicates_or_versions")
)

cross_server_outgoing_counts = (
    df[df["dedupe_behavior"] == "cross_server_reassigned"]
    .groupby("server_name")
    .size()
    .rename("cross_server_outgoing_records")
)

cross_server_incoming_counts = (
    df[df["dedupe_behavior"] == "cross_server_reassigned"]
    .groupby("parent_server_name_selected")
    .size()
    .rename("cross_server_incoming_records")
)

all_servers = sorted(
    set(before_counts.index)
    | set(after_counts.index)
    | set(stable_parent_counts.index)
    | set(internal_duplicate_counts.index)
    | set(cross_server_outgoing_counts.index)
    | set(cross_server_incoming_counts.index)
)

summary = pd.DataFrame(index=all_servers)

summary = summary.join(before_counts, how="left")
summary = summary.join(after_counts, how="left")
summary = summary.join(stable_parent_counts, how="left")
summary = summary.join(internal_duplicate_counts, how="left")
summary = summary.join(cross_server_incoming_counts, how="left")
summary = summary.join(cross_server_outgoing_counts, how="left")

summary = summary.fillna(0)

# ============================================================
# 5. METRICS
# ============================================================

summary["absolute_change_after_deduplication"] = (
    summary["parent_records_after_deduplication"]
    - summary["records_before_deduplication"]
)

summary["net_cross_server_flow"] = (
    summary["cross_server_incoming_records"]
    - summary["cross_server_outgoing_records"]
)

summary["pct_change_after_deduplication"] = np.where(
    summary["records_before_deduplication"] > 0,
    summary["absolute_change_after_deduplication"]
    / summary["records_before_deduplication"],
    np.nan,
)

summary["true_stability_rate"] = np.where(
    summary["records_before_deduplication"] > 0,
    summary["stable_parent_records"]
    / summary["records_before_deduplication"],
    np.nan,
)

summary["internal_duplication_rate"] = np.where(
    summary["records_before_deduplication"] > 0,
    summary["internal_duplicates_or_versions"]
    / summary["records_before_deduplication"],
    np.nan,
)

summary["cross_server_outgoing_rate"] = np.where(
    summary["records_before_deduplication"] > 0,
    summary["cross_server_outgoing_records"]
    / summary["records_before_deduplication"],
    np.nan,
)

summary["cross_server_incoming_rate"] = np.where(
    summary["parent_records_after_deduplication"] > 0,
    summary["cross_server_incoming_records"]
    / summary["parent_records_after_deduplication"],
    np.nan,
)

summary["overall_deduplication_reduction_rate"] = np.where(
    summary["records_before_deduplication"] > 0,
    (
        summary["records_before_deduplication"]
        - summary["parent_records_after_deduplication"]
    )
    / summary["records_before_deduplication"],
    np.nan,
)

# Sanity check:
# before = stable parents + internal duplicates + outgoing cross-server
summary["behavior_sum_check"] = (
    summary["stable_parent_records"]
    + summary["internal_duplicates_or_versions"]
    + summary["cross_server_outgoing_records"]
)

summary["behavior_sum_difference"] = (
    summary["records_before_deduplication"]
    - summary["behavior_sum_check"]
)

# ============================================================
# 6. SERVER TYPOLOGY
# ============================================================

def classify_server(row):
    stability = row["true_stability_rate"]
    internal_rate = row["internal_duplication_rate"]
    outgoing_rate = row["cross_server_outgoing_rate"]
    incoming = row["cross_server_incoming_records"]
    outgoing = row["cross_server_outgoing_records"]
    net = row["net_cross_server_flow"]

    if stability >= 0.90 and outgoing_rate < 0.05 and internal_rate < 0.10:
        return "Stable server"

    if internal_rate >= 0.20 and outgoing_rate < 0.10:
        return "Version-heavy within-server"

    if outgoing_rate >= 0.20 and incoming < outgoing:
        return "Donor / externally reassigned server"

    if incoming > outgoing * 1.5 and incoming > 0:
        return "Absorbing server"

    if incoming > 0 and outgoing > 0:
        return "Mixed-flow server"

    if stability < 0.50 and outgoing > 0:
        return "Mirror or repost-heavy server"

    return "Low-flow or unchanged server"

summary["server_typology"] = summary.apply(classify_server, axis=1)

# ============================================================
# 7. FLOW TABLES
# ============================================================

flow_table = (
    df.groupby(["server_name", "parent_server_name_selected"])
    .size()
    .reset_index(name="n_records")
    .rename(
        columns={
            "server_name": "from_server",
            "parent_server_name_selected": "to_server",
        }
    )
)

flow_table["flow_type"] = np.where(
    flow_table["from_server"] == flow_table["to_server"],
    "same_server",
    "cross_server",
)

from_totals = (
    flow_table.groupby("from_server")["n_records"]
    .sum()
    .rename("from_server_total")
)

flow_table = flow_table.merge(from_totals, on="from_server", how="left")

flow_table["pct_of_from_server"] = (
    flow_table["n_records"] / flow_table["from_server_total"]
)

# Cross-server only Sankey table
sankey_table = flow_table[flow_table["flow_type"] == "cross_server"].copy()

# ============================================================
# 8. HEATMAP MATRICES
# ============================================================

heatmap_counts = flow_table.pivot_table(
    index="from_server",
    columns="to_server",
    values="n_records",
    aggfunc="sum",
    fill_value=0,
)

heatmap_percent = heatmap_counts.div(
    heatmap_counts.sum(axis=1),
    axis=0,
).fillna(0)

# Cross-server only heatmap
cross_flow_table = flow_table[flow_table["flow_type"] == "cross_server"].copy()

cross_heatmap_counts = cross_flow_table.pivot_table(
    index="from_server",
    columns="to_server",
    values="n_records",
    aggfunc="sum",
    fill_value=0,
)

cross_heatmap_percent = cross_heatmap_counts.div(
    cross_heatmap_counts.sum(axis=1),
    axis=0,
).fillna(0)

# ============================================================
# 9. SIGNED MATRIX
# Positive = incoming to server
# Negative = outgoing from server
# ============================================================

signed_flows = cross_flow_table.copy()

signed_out = signed_flows.copy()
signed_out["signed_count"] = -signed_out["n_records"]
signed_out = signed_out.rename(
    columns={
        "from_server": "server_name",
        "to_server": "connected_server",
    }
)[["server_name", "connected_server", "signed_count"]]

signed_in = signed_flows.copy()
signed_in["signed_count"] = signed_in["n_records"]
signed_in = signed_in.rename(
    columns={
        "to_server": "server_name",
        "from_server": "connected_server",
    }
)[["server_name", "connected_server", "signed_count"]]

signed_long = pd.concat([signed_out, signed_in], ignore_index=True)

signed_matrix = signed_long.pivot_table(
    index="server_name",
    columns="connected_server",
    values="signed_count",
    aggfunc="sum",
    fill_value=0,
)

# ============================================================
# 10. SOURCE OF DEDUPLICATION TABLE
# ============================================================

dedupe_source_table = (
    df.groupby(["server_name", "dedupe_behavior"])
    .size()
    .reset_index(name="n_records")
)

dedupe_source_wide = dedupe_source_table.pivot_table(
    index="server_name",
    columns="dedupe_behavior",
    values="n_records",
    aggfunc="sum",
    fill_value=0,
)

dedupe_source_wide = dedupe_source_wide.reset_index()

# ============================================================
# 11. ARTICLE-READY SUMMARY TABLE
# ============================================================

summary = summary.reset_index().rename(columns={"index": "server_name"})

rate_cols = [
    "pct_change_after_deduplication",
    "true_stability_rate",
    "internal_duplication_rate",
    "cross_server_outgoing_rate",
    "cross_server_incoming_rate",
    "overall_deduplication_reduction_rate",
]

summary[rate_cols] = summary[rate_cols].round(4)

article_table = summary[
    [
        "server_name",
        "records_before_deduplication",
        "parent_records_after_deduplication",
        "absolute_change_after_deduplication",
        "pct_change_after_deduplication",
        "stable_parent_records",
        "internal_duplicates_or_versions",
        "cross_server_incoming_records",
        "cross_server_outgoing_records",
        "net_cross_server_flow",
        "true_stability_rate",
        "internal_duplication_rate",
        "cross_server_outgoing_rate",
        "cross_server_incoming_rate",
        "overall_deduplication_reduction_rate",
        "server_typology",
        "behavior_sum_difference",
    ]
].copy()

article_table = article_table.sort_values(
    by="records_before_deduplication",
    ascending=False,
)

# ============================================================
# 12. EXPORT FILES
# ============================================================

article_table.to_csv(
    OUT_DIR / "table_server_deduplication_impact_article_ready.csv",
    index=False,
)

summary.to_csv(
    OUT_DIR / "server_deduplication_metrics_full.csv",
    index=False,
)

flow_table.to_csv(
    OUT_DIR / "server_to_server_flow_long_all.csv",
    index=False,
)

sankey_table.to_csv(
    OUT_DIR / "server_to_server_flow_sankey_cross_server_only.csv",
    index=False,
)

heatmap_counts.to_csv(
    OUT_DIR / "server_flow_heatmap_counts_all.csv"
)

heatmap_percent.to_csv(
    OUT_DIR / "server_flow_heatmap_percent_all.csv"
)

cross_heatmap_counts.to_csv(
    OUT_DIR / "server_flow_heatmap_counts_cross_server_only.csv"
)

cross_heatmap_percent.to_csv(
    OUT_DIR / "server_flow_heatmap_percent_cross_server_only.csv"
)

signed_matrix.to_csv(
    OUT_DIR / "server_flow_signed_matrix.csv"
)

dedupe_source_wide.to_csv(
    OUT_DIR / "dedupe_source_by_server.csv",
    index=False,
)

# Excel package
with pd.ExcelWriter(OUT_DIR / "server_deduplication_impact_package.xlsx") as writer:
    article_table.to_excel(writer, sheet_name="article_ready_table", index=False)
    summary.to_excel(writer, sheet_name="full_metrics", index=False)
    flow_table.to_excel(writer, sheet_name="flow_all", index=False)
    sankey_table.to_excel(writer, sheet_name="sankey_cross_server", index=False)
    heatmap_counts.to_excel(writer, sheet_name="heatmap_counts_all")
    heatmap_percent.to_excel(writer, sheet_name="heatmap_percent_all")
    cross_heatmap_counts.to_excel(writer, sheet_name="heatmap_counts_cross")
    cross_heatmap_percent.to_excel(writer, sheet_name="heatmap_percent_cross")
    signed_matrix.to_excel(writer, sheet_name="signed_matrix")
    dedupe_source_wide.to_excel(writer, sheet_name="dedupe_source", index=False)

# ============================================================
# 13. DISPLAY QUICK CHECKS
# ============================================================

print("Done.")
print(f"Files saved in: {OUT_DIR.resolve()}")

print("\nTop 20 article-ready rows:")
print(article_table.head(20))

print("\nBehavior sum check:")
print(article_table["behavior_sum_difference"].describe())

print("\nDedupe behavior counts:")
print(df["dedupe_behavior"].value_counts())